# Personal Project Summer 2026
By Evan O'Malley

Context of the project  
- Recreational Project For Resumè Building
- Practice basic dsci and programming skill
- Dataset from Pew Research

Goals of the project  
- Focus on efficient and effective use of library functions
- Focus on making legible work, as if for presentation
- Focus on building a robust and coherent environment for analysis

## Initialization

### Notebook Setup  
Imports files and packages

In [1]:
# Necessary Imports
# Pretty Lean
import pandas as pd, numpy as np, statistics as st, matplotlib.pyplot as plt
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

rel_path = "Data/Western_Europe_Public_Data_Church_Tax_Added.csv"
codebook_rel_path = "Codebooks/codebook.txt"

### Precursor Structures
Paragraph

In [14]:
class OrderLegend:
    def __init__(self, order: str, legend: dict):
        self.order = order
        self.legend = legend
        
    def ord(self):
        return self.order
    
    def leg(self):
        return self.legend
    
    def values(self):
        return self.legend.values()

    def keys(self):
        return self.legend.keys()

    def items(self):
        return self.legend.items()

    def len(self):
        return len(self.legend)

    def __repr__(self):
        return f"{self.order}, {self.legend}"

    def __str__(self):
        return f"{self.order}, {self.legend}"

    def __getitem__(self, i):
        return self.legend[i]

def Legends_item(item):
    if item in Legends:
        return Legends[item]
    elif item[:-1] in Legends:
        return Legends[item[:-1]]

    raise KeyError

### File Parsing
Makes codebook.txt into usable object

In [3]:
with open(codebook_rel_path) as f:
    codebook = f.read()

def parse_next(terminator):
    assert len(terminator) == 1, "Terminator must be one character"
    global s
    global codebook
    
    passage = ""
    while codebook[s] != terminator:
        passage += codebook[s]
        s += 1
    s += 1
        
    return passage


s = 0
Legends = {}

while s < len(codebook):
    col_legend = {}
    name = parse_next(':')
    order = parse_next('{')
    s += 1
    
    if "[COUNTRY]" in name:
        while codebook[s] != '!':
            idx = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[idx] = val
        s += 1

        Legends[name.replace("[COUNTRY]", "")] = OrderLegend(order, {})
        
        while codebook[s] != '}':
            CTY = codebook[s:s+3]
            s += 5
            
            while codebook[s] != '}':
                idx = int(parse_next(' '))
                val = parse_next('\n')
                col_legend[idx] = val
            s += 2

            col_legend[98] = "Don't know"
            col_legend[99] = "Refused"

            Legends[name.replace("[COUNTRY]", CTY)
            ] = OrderLegend(order, col_legend)
            
        s += 2
            
    else:
        while codebook[s] != '}':
            idx = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[idx] = val
        s += 2

        col_legend[98] = "Don't know"
        col_legend[99] = "Refused"

        Legends[name] = OrderLegend(order, col_legend)

### Parent Restitching
These cells altar the parent DataFrame and cannot be rerun

In [4]:
Parent_DF = pd.read_csv(rel_path, skipinitialspace=True)

In [5]:
# Move QRID column to index
Parent_DF.set_index("QRID", inplace = True)

In [6]:
# Fix respose layout for Q9
q9_cols = Parent_DF.apply(lambda x: x.name[1] == "9")
q9_cols = Parent_DF.loc[:, q9_cols]

def get_Q9(row: pd.Series):
    i = row[row == 1].index
    if len(i) == 0:
        return np.nan
        
    else:
        return int(i[0][3])

Parent_DF.insert(16, "Q9", q9_cols.apply(get_Q9, axis = 1))
Parent_DF.drop(q9_cols, axis = 1, inplace = True)

In [7]:
# Removing QS1... variables becuase they stand for regions and are indecipherable or redacted
# Removing qbornmoverec variable because values are incomprehensible or redacted
QS1_cols = [i for i in Parent_DF.columns if i.lower()[:3] == "qs1"]

Parent_DF.drop(QS1_cols, axis = 1, inplace = True)
Parent_DF.drop("qbornmoverec", axis = 1, inplace = True)

In [8]:
# Knit QIDEOLOGY, QIDEOLOGYa, and QIDEOLOGYb,
# as they are nearly identical
ideology_rec = Parent_DF["QIDEOLOGY"].map(
               lambda x: x + 1 if x < 90 else x)
ideologya_rec = Parent_DF["QIDEOLOGYa"].map(
                lambda x: x + 1 if x< 90 else x)

Parent_DF["QIDEOLOGY"] = Parent_DF["QIDEOLOGYb"].fillna(
                         ideology_rec).fillna(ideologya_rec)

Parent_DF.drop(["QIDEOLOGYa", "QIDEOLOGYb"], axis = 1, inplace = True)

In [9]:
# Knit QDENOM[COUNTRY] columns
# as they use identical enumerations
qdenom = pd.Series([]).reindex(Parent_DF.index)
qdenom_cols = Parent_DF.loc[:, Parent_DF.apply(
              lambda x: x.name[:6].lower() == "qdenom")]

qdenom_cols.apply(lambda x: qdenom.fillna(x, inplace = True))
Parent_DF.insert(26, "QDenom", qdenom)

Parent_DF.drop(qdenom_cols, axis = 1, inplace = True)

In [10]:
# Same goes for QCHDENOM[Country] columns
qchdenom = pd.Series([]).reindex(Parent_DF.index)
qchdenom_cols = Parent_DF.loc[:, Parent_DF.apply(
                lambda x: x.name[:8].lower() == "qchdenom")]

qchdenom_cols.apply(lambda x: qchdenom.fillna(x, inplace = True))
Parent_DF.insert(27, "QChdenom", qchdenom)

Parent_DF.drop(qchdenom_cols, axis = 1, inplace = True)

In [11]:
# Same goes for QCURRELrec and QCURRELDrec columns
qcurrel = Parent_DF["QCURRELrec"].replace([91, 98, 99], np.nan)
Parent_DF["QCURRELrec"] = qcurrel.fillna(Parent_DF["QCURRELDrec"])

Parent_DF.drop("QCURRELDrec", axis = 1, inplace = True)

In [12]:
# Repair column naming scheme
def title_scheme(title: str):
    # Cumulatively adjusts column titles according to scheme described below
    new_title = title

    # Remove instances of "rec", signifying recoded variables

    if new_title[-3:].lower() == "rec":
        new_title = new_title[:-3]

    # Normalize Capitalization Scheme
    # "country" -> "Country"
    # "QCURREL", "qcurrel" -> "QCurrel"
    
    if new_title[0].lower() != 'q':
        new_title = new_title.title()
        
    else:
        new_title = 'Q' + new_title[1:].title()
    
    # Capitalize suffixes signifying country
    # "QDenomaut" -> "QDenomAUT"
    
    if new_title[-3:].upper() in Legends["Country"].values():
        new_title = new_title[:-3] + new_title[-3:].upper()

    if new_title[-4:-1].upper() in Legends["Country"].values():
        new_title = new_title[:-4] + new_title[-4:-1].upper() + new_title[-1]

    # Some questions are divided into cases a, b, c, etcetera
    # Denotation for this will be separated from the main title and uncapitalized
    # "Q4A", "Q4B" -> "Q4_a", Q4_b"
    # These are dicipherable by last 2 characters of the title

    NumCap = new_title[-2].isnumeric() and new_title[-1].isupper()
    CapUncap = new_title[-2].isupper() and new_title[-1].islower() 

    if NumCap or CapUncap:
        new_title = new_title[:-1] + "_" + new_title[-1].lower()

    # Choice adjustments
    if new_title[:4] == "QPty":
        if new_title[4] == 'a':
            new_title = new_title[:4] + "potvot" + new_title[5:]
        elif new_title[4] == 'b':
            new_title = new_title[:4] + "fvr" + new_title[5:]
        else:
            new_title = new_title[:4] + "cls" + new_title[4:]

    rename_key = {"QCitizen1" : "QCitizen",
                  "QBornc" : "QBornmthr",
                  "QBorne" : "QBornfthr",
                  "QChilda" : "QChild",
                  "QHhch" : "QParent"}

    if new_title in rename_key.keys():
        new_title = rename_key[new_title]
    
    return new_title

Parent_DF.rename(title_scheme, axis = 1, inplace = True)

In [15]:
# There are some entires that are undefined by the codebook
def constrain_values(col: pd.Series):
    if Legends_item(col.name).ord() == "Cardinal":
        return col
        
    return col.map(lambda x: x if x in Legends_item(col.name).leg() else np.nan)

Parent_DF = Parent_DF.apply(constrain_values)

### Further Preparations
Functions for table manipulations

In [16]:
# Cardinality function for titles
def cardinality(title: str):
    if title in Legends:
        return Legends[title].ord()
        
    if title[-2] == '_':
        return Legends[title[:-1]].ord()

    raise NotImplementedError(f"Column {title} inappropriately named")

In [17]:
# Function to divide tables into cardinalities
def SepOrder(df: pd.DataFrame, cardinality_arg: str):
    return df.loc[:, df.apply(
           lambda x: cardinality(x.name)) == cardinality_arg]

In [18]:
# Create Generalized list of Qs in order
# This will not be useful later
def strip_countries(title: str):
    # Code primary lifted from title_scheme function
    if title[-3:] in Legends["Country"].values():
        return title[:-3]

    if title[-5:-2] in Legends["Country"].values():
        return title[:-5] + title[-2:]

    return title
        
survey_order = Parent_DF.apply(
               lambda x: strip_countries(x.name)
               ).drop_duplicates().to_numpy()

In [19]:
# Function to disambiguate and ambiguate enumerations
def disambiguate(col: pd.Series):
    if col.dtypes in [int, float]:
        return col.dropna().map(lambda x: Legends[col.name][x])

    else:
        return col

def ambiguate(col: pd.Series):
    if col.dtypes in [int, float]:
        return col

    else:
        reverse_legend = {y:x for x,y in Legends[col.name].items()}
        return col.dropna().map(lambda x: reverse_legend[x])

In [20]:
# Function to divide tables into countries
# Returns a dictionary of the division
def SepCountry(refdf: pd.DataFrame):
    df = refdf.copy()
    country_dfs = {}
    if "Country" not in df:
        df = df.join(Parent_DF["Country"], how = "left")
    df["Country"] = disambiguate(df["Country"])

    for c in df["Country"].dropna().unique():
        country_df = df.copy()
        country_df = country_df.loc[country_df["Country"] == c]
        country_df = country_df.loc[:, country_df.apply(
                     lambda x: any(x.notna()))]

        country_df = country_df.rename(strip_countries, axis = 1)
        country_dfs[c] = country_df.drop("Country", axis = 1)
        
    return country_dfs

In [21]:
# Function to divide tables into belief systems
# Separates Christianity by denomination,
# due to overwhelmingly Christian responses
# Returns a dictionary of the division
def SepReligion(refdf: pd.DataFrame):
    df = refdf.copy()

    if "QCurrel" not in df:
        df = df.join(Parent_DF["QCurrel"], how = "left")
    df["QCurrel"] = disambiguate(df["QCurrel"])
    
    religion_dfs = {}
    for r in df["QCurrel"].dropna().unique():
        if r != "Christian":
            religion_df = df.copy()
            religion_df = religion_df.loc[religion_df["QCurrel"] == r]
            religion_df = religion_df.loc[:, religion_df.apply(
                          lambda x: any(x.notna()))]

            religion_dfs[r] = religion_df.drop("QCurrel", axis = 1)
    
    if "QDenom" not in df:
        df = df.join(Parent_DF["QDenom"], how = "left")
    df["QDenom"] = disambiguate(df["QDenom"])
    
    for d in df["QDenom"].dropna().unique():
        religion_df = df.copy()
        religion_df = religion_df.loc[religion_df["QDenom"] == d]
        religion_df = religion_df.loc[:, religion_df.apply(
                      lambda x: any(x.notna()))]

        religion_dfs[d] = religion_df.drop(["QCurrel", "QDenom"], axis = 1)
            
    return religion_dfs

### Initialization Summary
Coagulate essenstial information  
- Legends object  
- OrderLegend class and methods

Prepare Parent DataFrame  
- Column cropping  
- Rename scheme

Create DataFrame editing tools  
I have not used any of these meaningfully yet  
- SepOrder  
- SepNation
- SepReligion
- Supplementary functions

## Exploratory Analysis

### Statistics Invention
Start inventing some summary statistics

In [22]:
def hist_areas(dfref = pd.DataFrame):
    df = dfref.copy()
    def _responses(x):
        responses = x.value_counts()
        responses[98] = 0
        responses[99] = 0
        responses.drop([98, 99], inplace = True)
        return responses / sum(responses)
        
    return df.apply(_responses)

In [81]:
# Compare response distributions across countries
Ordinal_DF = SepOrder(Parent_DF, "Ordinal")
Ordinal_DF["Country"] = disambiguate(Parent_DF["Country"])
gb_nation = Ordinal_DF.groupby("Country").apply(hist_areas)

def compare_hist_areas(col: pd.Series, ref_idx = 0):
    frame = Legends_item(col.name).len() - 2

    comp_col = col.groupby(level = 0)
    comp_col = comp_col.apply(lambda x: x.dropna().to_numpy())
    comp_col = comp_col.apply(lambda x: np.pad(x, (0, frame - len(x))))

    ref_arr = col[comp_col.index[ref_idx]].dropna().to_numpy()
    ref_arr = np.pad(ref_arr, (0, frame - len(ref_arr)))
    
    return comp_col.apply(lambda x: sum(np.minimum(x, ref_arr)))

gb_nation.apply(compare_hist_areas).loc[
:, gb_nation.apply(lambda x: x.name == strip_countries(x.name))]

,Q2,Q5,Q6,Q10_a,Q10_b,Q10_c,Q15_a,Q15_b,Q16,Q20_a,Q20_b,Q20_c,Q20_d,Q20_e,Q20_f,Q20_g,Q20_h,Q24,Q27,Q32_a,Q32_b,Q33,Q42_a,Q42_b,Q42_c,Q42_d,Q43,Q47,Q48_a,Q48_b,Q48_c,Q49_a,Q49_b,Q49_c,Q49_d,Q49_e,Q50,QIdeology,Isced
Country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
AUT,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
BEG,0.912783,0.739141,0.807039,0.811136,0.932970,0.816199,0.846356,0.604070,0.877903,0.893528,0.935946,0.719498,0.649929,0.853107,0.882463,0.835247,0.894706,0.868631,0.714059,0.752191,0.722619,0.921754,0.768247,0.817466,0.825607,0.889610,0.799221,0.736463,0.907745,0.899692,0.873917,0.904572,0.966174,0.785229,0.928959,0.848201,0.707789,0.747831,0.737477
CHE,0.929887,0.984330,0.958335,0.974793,0.951577,0.952805,0.943975,0.705556,0.969337,0.921411,0.944061,0.962618,0.977164,0.947961,0.922209,0.980146,0.981185,0.965843,0.940064,0.964590,0.810714,0.944948,0.978356,0.973827,0.984925,0.976223,0.930871,0.973612,0.943851,0.959981,0.956871,0.963750,0.956812,0.953797,0.925520,0.964597,0.969273,0.975544,0.998106
DEU,0.985936,0.967992,0.975231,0.965540,0.966588,0.967363,0.952438,0.833803,0.957028,0.934407,0.948349,0.936781,0.954673,0.955877,0.956090,0.979396,0.985511,0.959126,0.974197,0.951872,0.834859,0.937076,0.973672,0.976827,0.967788,0.988550,0.981972,0.974925,0.872887,0.949164,0.945710,0.934455,0.987319,0.933555,0.974936,0.898060,0.964670,0.907190,0.981445
DNK,0.905837,0.700111,0.614142,0.852030,0.911564,0.913868,0.811598,0.796429,0.814658,0.805666,0.850039,0.708905,0.739981,0.867557,0.822116,0.812877,0.886707,0.879171,0.597604,0.702736,0.867647,0.910246,0.968495,0.858838,0.974578,0.860792,0.818341,0.888380,0.945147,0.877606,0.883316,0.819532,0.898740,0.826279,0.939564,0.783588,0.897223,0.808609,0.709566
ESP,0.705166,0.848172,0.840114,0.858462,0.940203,0.765894,0.834775,0.642391,0.853492,0.875701,0.901195,0.818675,0.718309,0.892986,0.920701,0.937244,0.907883,0.793599,0.604572,0.765379,0.443478,0.894739,0.760109,0.820315,0.785583,0.945763,0.805202,0.811957,0.863815,0.836606,0.813970,0.804828,0.909211,0.837378,0.966952,0.832613,0.673372,0.812702,0.697381
FIN,0.903279,0.804006,0.705718,0.896389,0.879866,0.883848,0.728821,0.250000,0.869318,0.771999,0.888809,0.844744,0.770390,0.936767,0.903110,0.889862,0.967234,0.762194,0.631427,0.764879,0.200000,0.904468,0.872062,0.909892,0.933951,0.718407,0.937116,0.860159,0.833536,0.844788,0.872971,0.946857,0.673347,0.930238,0.951869,0.921002,0.869349,0.744962,0.682339
FRA,0.927514,0.978573,0.894652,0.904220,0.969681,0.944500,0.895705,0.858333,0.936643,0.944465,0.930077,0.932695,0.911954,0.926374,0.930457,0.918488,0.943668,0.938952,0.922921,0.930337,0.825971,0.921910,0.944977,0.919340,0.931855,0.915204,0.965724,0.959379,0.957958,0.983419,0.954171,0.965190,0.971300,0.940663,0.985788,0.983644,0.908200,0.908153,0.900909
GBR,0.975136,0.989184,0.959705,0.989649,0.979942,0.983008,0.914804,0.847727,0.914493,0.902458,0.928036,0.939432,0.924792,0.933447,0.956270,0.955391,0.938785,0.940020,0.971710,0.923688,0.752273,0.918572,0.956606,0.922925,0.920956,0.932853,0.960264,0.965026,0.913626,0.909681,0.949281,0.957361,0.975988,0.969336,0.975770,0.956083,0.952087,0.932999,0.693805
